In [1]:
import torch
import torchvision
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import xml.etree.ElementTree as ET
import os

HOME = os.path.abspath(os.sep)
dataset_folder = os.getcwd() + "/Dataset"

In [2]:
class GunDataset(torch.utils.data.Dataset):
  def __init__(self, root_dir, transform=None):
    self.root_dir = root_dir
    self.transform = transform
    self.imgs = list(sorted(os.listdir(os.path.join(root_dir, "Images"))))
    self.annots = list(sorted(os.listdir(os.path.join(root_dir, "Annotations"))))

  def __len__(self):
    return len(self.imgs)

  def __getitem__(self, idx):
    img_path = os.path.join(self.root_dir, "Images", self.imgs[idx])
    img = Image.open(img_path).convert("RGB")

    annot_path = os.path.join(self.root_dir, "Annotations", self.annots[idx])

    xml_tree = ET.parse(annot_path)
    root = xml_tree.getroot()

    boxes = []
    labels = []

    for obj in root.findall("object"):
      label = obj.find("name").text
      if label == "gun":
        bbox = obj.find("bndbox")
        xmin = int(bbox.find("xmin").text)
        ymin = int(bbox.find("ymin").text)
        xmax = int(bbox.find("xmax").text)
        ymax = int(bbox.find("ymax").text)
        boxes.append([xmin, ymin, xmax, ymax])
        labels.append(1) # gun label

    if len(boxes) == 0:
        boxes = torch.empty((0, 4), dtype=torch.float32)  # Forma [0, 4]
        labels = torch.empty((0,), dtype=torch.int64)     # Nessuna etichetta

    else:
        boxes = torch.as_tensor(boxes, dtype=torch.float32)
        labels = torch.as_tensor(labels, dtype=torch.int64)


    target = {}
    target["boxes"] = boxes
    target["labels"] = labels

    if self.transform:
      img = self.transform(img)
    return img, target

In [3]:
import torchvision.transforms as T
from torch.utils.data import DataLoader, random_split


# image transformations
transform = T.Compose([
    T.ToTensor(),
])

# load dataset
dataset = GunDataset(dataset_folder, transform=transform)

# define splits
train_size = int(0.8*len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

num_workers = 2
#num_workers
if os.name == 'nt':
    num_workers = 0

# data loaders
train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True, num_workers=num_workers, collate_fn=lambda x: tuple(zip(*x)), pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=2, shuffle=False, num_workers=num_workers, collate_fn=lambda x: tuple(zip(*x)))

In [4]:
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FasterRCNN_ResNet50_FPN_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

model = fasterrcnn_resnet50_fpn(weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT)
num_classes = 2
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

Downloading: "https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth" to C:\Users\Gianmarco/.cache\torch\hub\checkpoints\fasterrcnn_resnet50_fpn_coco-258fb6c6.pth
100%|██████████| 160M/160M [00:03<00:00, 54.3MB/s] 


In [5]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
model.to(device)

optimizer = torch.optim.SGD(model.parameters(), lr=0.005, momentum=0.9, weight_decay=0.0005)
lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size = 3, gamma = 0.1)

In [8]:
from torch.utils.tensorboard import SummaryWriter

# Inizializza il writer di TensorBoard
writer = SummaryWriter()

num_epochs = 4

for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for images, targets in train_loader:
        # Trasferisci le immagini e i target nel dispositivo
        images = list(image.to(device) for image in images)
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        # Calcola la loss
        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())

        # Ottimizzazione
        optimizer.zero_grad()
        losses.backward()
        optimizer.step()

        total_loss += losses.item()

    # Aggiungi la perdita (loss) di questa epoca a TensorBoard
    writer.add_scalar('Loss/train', total_loss, epoch)

    print(f"Epoca: {epoch + 1}, Loss: {total_loss}")
    
    # Aggiorna il learning rate scheduler
    lr_scheduler.step()

# Alla fine dell'addestramento, stampa i pesi finali
print("Pesi finali del classificatore (cls_score):")
print(model.roi_heads.box_predictor.cls_score.weight)

# Chiudi il writer di TensorBoard
writer.close()

Epoca: 1, Loss: 0.2751572395444697
Epoca: 2, Loss: 0.006996756923669523
Epoca: 3, Loss: 0.005221848678701235
Epoca: 4, Loss: 0.004780255058065563
Epoca: 5, Loss: 0.004436879368427071
Epoca: 6, Loss: 0.0048594171903744154
Epoca: 7, Loss: 0.004542096318601807
Epoca: 8, Loss: 0.004839407222576142
Epoca: 9, Loss: 0.004803338826604886
Epoca: 10, Loss: 0.0047045994180621165
Pesi finali del classificatore (cls_score):
Parameter containing:
tensor([[ 0.0009,  0.0073,  0.0062,  ...,  0.0228, -0.0003,  0.0267],
        [-0.0254, -0.0014, -0.0044,  ...,  0.0080, -0.0068,  0.0006]],
       device='cuda:0', requires_grad=True)
